# Warehouse Operations Simulation and Optimization Using SimPyThis notebook implements a discrete-event simulation of warehouse operations.It models receiving, storage, order picking, packing, and shipping processes.## Run all cells sequentially to execute 11 experimental scenarios (Phase 2) and view KPI reports.

In [ ]:
!pip install simpy pandas numpy matplotlib statsmodels scikit-learn joblib -qprint("Dependencies installed")

---## 1. Imports & Configuration

In [ ]:
import sys, ossys.path.insert(0, os.path.dirname(os.path.abspath('.')))from src.config import (    SimulationConfig, Order, Product, Shift, WorkerProfile,    create_default_products, ORDER_PENDING, ORDER_PICKING,    ORDER_PACKING, ORDER_QUALITY_CHECK, ORDER_SHIPPING, ORDER_COMPLETED)from src.order_generator import OrderGeneratorfrom src.inventory import InventoryManagerfrom src.worker import WorkerPoolfrom src.picking import PickingProcessfrom src.packing import PackingProcessfrom src.shipping import ShippingProcessfrom src.warehouse import WarehouseSimulationfrom src.analytics import Analyticsfrom src.costing import CostConfig, CostTrackerfrom src.agv import AGVManagerfrom src.robot_picker import RobotPoolfrom src.forecasting import DemandForecasterfrom src.scheduler import WorkerScheduler%matplotlib inlineimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom IPython.display import displayprint("All modules loaded")

---## 2. Order Generator

In [ ]:
# Imported from src/order_generator.pyprint("OrderGenerator loaded")

---## 3. Inventory Management

In [ ]:
# Imported from src/inventory.pyprint("InventoryManager loaded")

---## 4. Worker Pool

In [ ]:
# Imported from src/worker.py (includes Phase 1: breaks, variable speeds)print("WorkerPool loaded")

---## 5. Picking Process

In [ ]:
# Imported from src/picking.py (includes AGV and robot picking time methods)print("PickingProcess loaded")

---## 6. Packing Process

In [ ]:
# Imported from src/packing.pyprint("PackingProcess loaded")

---## 7. Shipping Process

In [ ]:
# Imported from src/shipping.pyprint("ShippingProcess loaded")

---## 8. Main Warehouse Simulation Engine

In [ ]:
# Imported from src/warehouse.py (includes Phase 2: CostTracker, AGVManager, RobotPool, forecast_rates)print("WarehouseSimulation loaded")

---## 9. Analytics & KPI Engine

In [ ]:
# Imported from src/analytics.py (includes cost KPIs, AGV/robot utilization, cost charts)print("Analytics loaded")

---## 10. Scenario Runner

In [ ]:
def run_scenario(config, name):    sim = WarehouseSimulation(config)    sim.run()    total_time = config.working_hours * 60 * config.simulation_days    worker_util = sim.workers.get_utilization(total_time)    cost_summary = sim.cost_tracker.get_summary()    agv_util = sim.agv_manager.get_utilization(total_time) if config.num_agvs > 0 else None    robot_util = sim.robot_pool.get_utilization(total_time) if config.num_robot_pickers > 0 else None    analytics = Analytics(config)    kpis = analytics.compute_kpis(        sim.orders, sim.completed_orders, total_time,        worker_util, cost_summary, agv_util, robot_util    )    analytics.print_report(kpis, name)    df = analytics.create_dataframe(sim.completed_orders)    analytics.plot_results(kpis, df, name, show=True)    analytics.plot_costs(cost_summary, name, show=True)    return kpis, dfprint("Scenario runner ready")

---## 11. Run All 11 ScenariosExecutes 5 baseline + 6 Phase 2 scenarios (AGV, Robot, Forecasting, ML Scheduling).

In [ ]:
print("=" * 60)print("  WAREHOUSE OPERATIONS SIMULATION (PHASE 2)")print("  Using SimPy Discrete-Event Simulation")print("=" * 60)all_results = []scenarios = [    (SimulationConfig(num_workers=5,  num_forklifts=2, num_packing_stations=3),                "Scenario 1 - Current Config"),    (SimulationConfig(num_workers=7,  num_forklifts=2, num_packing_stations=3),                "Scenario 2 - More Workers"),    (SimulationConfig(num_workers=5,  num_forklifts=2, num_packing_stations=5),                "Scenario 3 - More Packing Stations"),    (SimulationConfig(num_workers=5,  num_forklifts=2, num_packing_stations=3, orders_per_hour=30.0), "Scenario 4 - 50% More Demand"),    (SimulationConfig(num_workers=5,  num_forklifts=4, num_packing_stations=3),                "Scenario 5 - More Forklifts"),    (SimulationConfig(num_workers=3,  num_agvs=2, agv_mode="mixed"),                           "Scenario 6 - Human + AGV Mixed"),    (SimulationConfig(num_workers=2,  num_agvs=3, agv_mode="agv_only"),                        "Scenario 7 - AGV Only"),    (SimulationConfig(num_workers=3,  num_robot_pickers=2, robot_mode="default"),              "Scenario 8 - Human + Robot Mixed"),    (SimulationConfig(num_workers=1,  num_robot_pickers=4, robot_mode="default",                      num_agvs=2, agv_mode="agv_only"),                                        "Scenario 9 - Full Automation"),]for config, name in scenarios:    kpis, df = run_scenario(config, name)    all_results.append((name, kpis))# Scenario 10: Forecast-Driven Demandfc_rates = [15, 18, 22, 25, 28, 25, 20, 16]fc_config = SimulationConfig(enable_forecasting=True, working_hours=8.0, random_seed=99, num_workers=5)fc_sim = WarehouseSimulation(fc_config, forecast_rates=fc_rates)fc_sim.run()total_time = fc_config.working_hours * 60worker_util = fc_sim.workers.get_utilization(total_time)cost_summary = fc_sim.cost_tracker.get_summary()analytics = Analytics(fc_config)fc_kpis = analytics.compute_kpis(fc_sim.orders, fc_sim.completed_orders, total_time, worker_util, cost_summary)analytics.print_report(fc_kpis, "Scenario 10 - Forecast-Driven Demand")df_fc = analytics.create_dataframe(fc_sim.completed_orders)analytics.plot_results(fc_kpis, df_fc, "Scenario 10 - Forecast-Driven Demand", show=True)analytics.plot_costs(cost_summary, "Scenario 10 - Forecast-Driven Demand", show=True)all_results.append(("Scenario 10 - Forecast-Driven Demand", fc_kpis))# Scenario 11: ML Schedulerml_config = SimulationConfig(orders_per_hour=25.0, working_hours=4.0, num_workers=8,                              enable_ml_scheduling=True, ml_scheduling_training_runs=10, random_seed=77)scheduler = WorkerScheduler(ml_config)train_data = scheduler.generate_training_data(num_runs=10)scheduler.train(train_data)recommended = scheduler.predict_optimal_workers(ml_config)ml_config.num_workers = recommendedml_sim = WarehouseSimulation(ml_config)ml_sim.run()total_time = ml_config.working_hours * 60worker_util = ml_sim.workers.get_utilization(total_time)cost_summary = ml_sim.cost_tracker.get_summary()analytics = Analytics(ml_config)ml_kpis = analytics.compute_kpis(ml_sim.orders, ml_sim.completed_orders, total_time, worker_util, cost_summary)analytics.print_report(ml_kpis, f"Scenario 11 - ML Scheduler ({recommended} workers)")df_ml = analytics.create_dataframe(ml_sim.completed_orders)analytics.plot_results(ml_kpis, df_ml, "Scenario 11 - ML Scheduler", show=True)analytics.plot_costs(cost_summary, "Scenario 11 - ML Scheduler", show=True)all_results.append((f"Scenario 11 - ML Scheduler ({recommended} workers)", ml_kpis))print("\nAll 11 scenarios complete")

---## 12. Scenario Comparison Table

In [ ]:
print("\n" + "=" * 100)print("  SCENARIO COMPARISON SUMMARY (PHASE 2)")print("=" * 100)print(f"{'Scenario':<30} {'Completed':<10} {'Avg Time':<10} {'Worker%':<8} {'AGV%':<7} {'Robot%':<8} {'Throughput':<10} {'Total Cost':<12} {'Cost/Order':<10}")print("-" * 100)for name, kpis in all_results:    cost = kpis.get('cost_total_cost', 0)    cpo = kpis.get('cost_cost_per_order', 0)    wu = kpis.get('worker_utilization', 0)    au = kpis.get('agv_utilization', None)    ru = kpis.get('robot_utilization', None)    au_str = f"{au:<6.1f}%" if au is not None else "N/A    "    ru_str = f"{ru:<6.1f}%" if ru is not None else "N/A    "    print(f"{name:<30} {kpis.get('orders_completed', 0):<10} {kpis.get('avg_completion_time', 0):<10.2f} {wu:<7.1f}% {au_str} {ru_str} {kpis.get('throughput_per_day', 0):<10} ${cost:<8.2f} ${cpo:<6.2f}")print("=" * 100)comparison_data = []for name, kpis in all_results:    comparison_data.append({        'Scenario': name,        'Orders Completed': kpis.get('orders_completed', 0),        'Avg Time (min)': round(kpis.get('avg_completion_time', 0), 2),        'Worker Util (%)': round(kpis.get('worker_utilization', 0), 1),        'AGV Util (%)': round(kpis.get('agv_utilization', 0), 1) if kpis.get('agv_utilization') else None,        'Robot Util (%)': round(kpis.get('robot_utilization', 0), 1) if kpis.get('robot_utilization') else None,        'Total Cost ($)': round(kpis.get('cost_total_cost', 0), 2),        'Cost/Order ($)': round(kpis.get('cost_cost_per_order', 0), 2),    })comparison_df = pd.DataFrame(comparison_data)display(comparison_df)

---## 13. Combined KPI Comparison Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))names = [r[0] for r in all_results]names_short = [n.split("-")[-1].strip() for n in names]x = range(len(names))completed = [r[1].get('orders_completed', 0) for r in all_results]axes[0].bar(x, completed, color='steelblue', edgecolor='black')axes[0].set_xticks(x); axes[0].set_xticklabels(names_short, rotation=45, ha='right', fontsize=9)axes[0].set_ylabel('Orders'); axes[0].set_title('Orders Completed')avg_time = [r[1].get('avg_completion_time', 0) for r in all_results]colors = ['coral' if t > 20 else 'green' for t in avg_time]axes[1].bar(x, avg_time, color=colors, edgecolor='black')axes[1].set_xticks(x); axes[1].set_xticklabels(names_short, rotation=45, ha='right', fontsize=9)axes[1].set_ylabel('Minutes'); axes[1].set_title('Avg Completion Time')axes[1].axhline(y=20, color='r', linestyle='--', alpha=0.5, label='Threshold'); axes[1].legend()cpo = [r[1].get('cost_cost_per_order', 0) for r in all_results]axes[2].bar(x, cpo, color='gold', edgecolor='black')axes[2].set_xticks(x); axes[2].set_xticklabels(names_short, rotation=45, ha='right', fontsize=9)axes[2].set_ylabel('$'); axes[2].set_title('Cost per Order')plt.suptitle('Warehouse Simulation - Scenario Comparison (Phase 2)', fontsize=14)plt.tight_layout(); plt.show()